# Система обнаружения стеганографически скрытой информации в цифровых и мультимедийных файлах с помощью нейросетей


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Импорт библиотек
import re
import json
import math
import random
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve


In [ ]:
# Установка зависимостей

!pip -q install --upgrade pip

!pip -q install \
  torch torchvision \
  pillow \
  albumentations \
  opencv-python-headless \
  tqdm \
  pandas \
  scikit-learn \
  matplotlib


In [ ]:
import os
from pathlib import Path

CANDIDATE_ROOTS = [Path("/content/drive/MyDrive/FQW/Stego-Images-Dataset_archive"), Path("/content/drive/MyDrive/FQW/Stego-Images-Dataset_archive")]
DRIVE_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)

if DRIVE_ROOT is None:
    raise RuntimeError("Не найден корень Google Drive. Проверьте, что drive.mount() выполнился успешно.")

print("DRIVE_ROOT =", DRIVE_ROOT)
print("Папок в корне (первые 10):")
print(sorted([x.name for x in DRIVE_ROOT.iterdir()])[:10])


DRIVE_ROOT = /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive
Папок в корне (первые 10):
['test', 'train', 'val']


In [ ]:
# Папка с набором данных
from collections import deque

DATASET_FOLDER_NAME = "Stego-Images-Dataset_archive"
MAX_DEPTH = 6  # ограничение глубины поиска

def find_folder_by_name(root: Path, folder_name: str, max_depth: int = 6):
    """
    Аккуратный BFS-поиск папки по имени с ограничением глубины.
    Возвращает Path или None.
    """
    q = deque([(root, 0)])
    while q:
        cur, d = q.popleft()
        if cur.name == folder_name and cur.is_dir():
            return cur
        if d >= max_depth:
            continue
        try:
            for child in cur.iterdir():
                if child.is_dir():
                    q.append((child, d + 1))
        except PermissionError:
            continue
    return None

dataset_root = DRIVE_ROOT / DATASET_FOLDER_NAME
if not dataset_root.exists():
    print("Папка по умолчанию не найдена:", dataset_root)
    print("Авто-поиск в пределах глубины", MAX_DEPTH)
    found = find_folder_by_name(DRIVE_ROOT, DATASET_FOLDER_NAME, max_depth=MAX_DEPTH)
    if found is None:
        print("Не нашёл автоматически. Укажите путь вручную ниже.")
        dataset_root = None
    else:
        dataset_root = found

print("dataset_root =", dataset_root)


Папка по умолчанию не найдена: /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/Stego-Images-Dataset_archive
Авто-поиск в пределах глубины 6
dataset_root = /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive


In [ ]:
# Выведим первые PNG-файлы внутри разделенных каталогов
import itertools

def sample_pngs(root: Path, n=5):
    if not root.exists():
        return []
    it = root.rglob("*.png")
    return [str(p) for p in itertools.islice(it, n)]

for split in ["train", "val", "test"]:
    split_dir = dataset_root / split
    print(f"\nSample PNGs under {split_dir}:")
    for p in sample_pngs(split_dir, n=5):
        print(" ", p)



Sample PNGs under /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/train:
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/train/train/stego/image_03724_url_0.png
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/train/train/stego/image_03719_ps_0.png
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/train/train/stego/image_03711_eth_0.png
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/train/train/stego/image_03686_url_0.png
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/train/train/stego/image_03712_html_0.png

Sample PNGs under /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/val:
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/val/val/clean/06794.png
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/val/val/clean/06246.png
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/val/val/clean/06893.png
  /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/val/val/clean/06800.png
  /content/drive/MyD

In [ ]:
# Разрешенные расширения изображений для стеганографии

ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".pgm", ".ppm", ".webp"}

# Структура датасета: разделы (train/val/test) и подклассы (чистые/стего изображения)

SPLIT_LAYOUT = {
    "train": {"clean": 0, "stego": 1},
    "val":   {"clean": 0, "stego": 1},
    "test":  {"clean": 0, "stego": 1, "stego_b64": 1, "stego_zip": 1},
}

def list_images_recursive(folder: Path, allowed_exts=ALLOWED_EXTS) -> List[Path]:
    """Рекурсивный поиск всех изображений в папке с проверкой расширений"""
    if folder is None or (not folder.exists()):
        return []
    files = []
    for p in folder.rglob("*"):  # rglob = recursive glob (**/pattern)
        if p.is_file() and p.suffix.lower() in allowed_exts:
            files.append(p)
    return sorted(files)

def default_group_id(path: Path) -> str:
    """Извлечение идентификатора группы из имени файла (без расширения)"""
    return path.stem

def resolve_split_dir(dataset_root: Path, split: str, max_depth: int = 5) -> Path:
    """
    Находит фактический корень split, проваливаясь в повторяющиеся подпапки:
      dataset_root/train/train/... -> вернёт dataset_root/train/train
    """
    d = dataset_root / split
    steps = 0
    while steps < max_depth:
        nxt = d / split
        if nxt.exists() and nxt.is_dir():
            d = nxt
            steps += 1
        else:
            break
    return d

def build_index_fixed(dataset_root: Path) -> pd.DataFrame:
    """Построение индекса датасета стеганографии"""
    rows = []
    diag = []   # для печати найденных папок

    # Обход всех разделов и подклассов согласно структуре
    for split, subclass_map in SPLIT_LAYOUT.items():
        split_dir = resolve_split_dir(dataset_root, split)

        for subclass, label in subclass_map.items():
            folder = split_dir / subclass

            # если строгий путь не найден — оставим как есть (ниже покажем диагностику)
            if not folder.exists():
                diag.append((split, subclass, "MISSING", str(folder)))
                continue

            img_paths = list_images_recursive(folder)
            diag.append((split, subclass, f"OK images={len(img_paths)}", str(folder)))

            # Формирование записей для датафрейма
            for p in img_paths:
                rows.append({
                    "split": split,
                    "subclass": subclass,
                    "label": int(label),
                    "path": str(p),
                    "ext": p.suffix.lower(),
                    "stem": p.stem,
                    "group_id": default_group_id(p),
                })

    # Диагностика того, что именно нашлось/не нашлось
    print("\n[DIAG] split/subclass folders:")
    for split, subclass, status, path in diag:
        print(f"  - {split}/{subclass}: {status} -> {path}")

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError(
            "Не найдено ни одного изображения даже после resolve_split_dir().\n"
            "Возможные причины: (1) папки классов названы иначе; (2) расширения не в ALLOWED_EXTS;"
            "(3) данные лежат не в clean/stego, а глубже/в других именах."
        )
    return df

# Построение индекса и вывод статистики
index_df = build_index_fixed(dataset_root)
print("\nOK. Всего файлов:", len(index_df))
index_df.head()



[DIAG] split/subclass folders:
  - train/clean: OK images=4000 -> /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/train/train/clean
  - train/stego: OK images=12000 -> /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/train/train/stego
  - val/clean: OK images=2000 -> /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/val/val/clean
  - val/stego: OK images=6000 -> /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/val/val/stego
  - test/clean: OK images=2000 -> /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/test/test/clean
  - test/stego: OK images=6000 -> /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/test/test/stego
  - test/stego_b64: OK images=6000 -> /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/test/test/stego_b64
  - test/stego_zip: OK images=6000 -> /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive/test/test/stego_zip

OK. Всего файлов: 44000


,split,subclass,label,path,ext,stem,group_id
0,train,clean,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...,.png,00001,00001
1,train,clean,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...,.png,00002,00002
2,train,clean,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...,.png,00003,00003
3,train,clean,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...,.png,00004,00004
4,train,clean,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...,.png,00005,00005


In [ ]:
import nbformat
import os

# Исправленный путь: без слэша в конце
notebook_path = '/content/drive/MyDrive/FQW/SRNet_LSB_2bit_grayscale_256.ipynb'

if os.path.exists(notebook_path):
    # Читаем ноутбук
    with open(notebook_path, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    # Удаляем проблемные метаданные виджетов
    if 'widgets' in nb.metadata:
        del nb.metadata['widgets']
        print(" Ключ 'metadata.widgets' успешно удален.")
    else:
        print("ℹ Ключ 'metadata.widgets' не найден, метаданные в порядке.")

    # Сохраняем исправленный ноутбук
    with open(notebook_path, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)

    print(f" Ноутбук сохранен. Можно загружать на GitHub.")
else:
    print(f" Файл не найден по пути: {notebook_path}")
    print(" Проверьте, что Google Drive подключен:")
    print("   from google.colab import drive")
    print("   drive.mount('/content/drive')")